<a href="https://colab.research.google.com/github/faisu6339-glitch/Deep-Learning/blob/main/LeNet_5.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## LeNet-5 Architecture Explained

LeNet-5 is a pioneering convolutional neural network (CNN) developed by Yann LeCun et al. in 1998 for handwritten digit recognition, specifically for reading zip codes and checks. It was one of the first successful applications of CNNs and laid the foundation for modern deep learning architectures.

### Key Characteristics:

1.  **End-to-End Learning:** LeNet-5 processes raw pixel input directly, learning features automatically rather than relying on hand-crafted features.
2.  **Local Receptive Fields:** Neurons in a convolutional layer are connected only to a small region of the input, allowing for localized feature detection.
3.  **Shared Weights:** Each filter (kernel) in a convolutional layer is applied across the entire input, sharing the same weights. This reduces the number of parameters and makes the model more efficient.
4.  **Pooling (Subsampling) Layers:** These layers reduce the spatial dimensions of the feature maps, making the network more robust to small shifts and distortions in the input and reducing computational cost.
5.  **Hierarchical Feature Extraction:** The network learns increasingly complex features through multiple layers of convolutions and pooling.

### LeNet-5 Layers (Simplified View for MNIST 32x32 Input):

LeNet-5 typically processes grayscale images of size 32x32 pixels.

*   **Input Layer (Input):** A 32x32 pixel grayscale image.

*   **Layer C1 (Convolutional Layer):**
    *   6 feature maps (channels).
    *   Each feature map is 28x28 pixels.
    *   Uses 5x5 kernels (filters).
    *   Stride of 1.
    *   Number of parameters: (5*5 + 1) * 6 = 156 (25 weights + 1 bias per filter, 6 filters).

*   **Layer S2 (Subsampling/Pooling Layer):**
    *   6 feature maps.
    *   Each feature map is 14x14 pixels (reduced by 2x2 average pooling).
    *   Uses 2x2 average pooling filters.
    *   Stride of 2.
    *   Each unit takes the average of 2x2 neighboring units in the corresponding C1 feature map and multiplies it by a learnable coefficient, then adds a learnable bias, and passes the result through an activation function. This effectively allows the network to learn to scale and shift its output.
    *   Number of parameters: (1 + 1) * 6 = 12 (1 weight + 1 bias per feature map).

*   **Layer C3 (Convolutional Layer):**
    *   16 feature maps.
    *   Each feature map is 10x10 pixels.
    *   Uses 5x5 kernels.
    *   Stride of 1.
    *   This layer connects to subsets of S2's feature maps (a sparse connection table was used to keep parameter count low, which was common before full connectivity became computationally feasible). In modern implementations, it's often fully connected to all S2 feature maps.
    *   Number of parameters (modern, full connection): (5*5*6 + 1) * 16 = 2416.

*   **Layer S4 (Subsampling/Pooling Layer):**
    *   16 feature maps.
    *   Each feature map is 5x5 pixels (reduced by 2x2 average pooling).
    *   Uses 2x2 average pooling filters.
    *   Stride of 2.
    *   Similar to S2, with learnable coefficients and biases.
    *   Number of parameters: (1 + 1) * 16 = 32.

*   **Layer C5 (Fully Connected/Convolutional Layer):**
    *   120 feature maps.
    *   Each feature map is 1x1 pixels (this is effectively a fully connected layer when applied to a 5x5 input using 5x5 kernels).
    *   Uses 5x5 kernels.
    *   Connects all 16 of the S4 feature maps to each of the 120 C5 neurons.
    *   Number of parameters: (5*5*16 + 1) * 120 = 48120.

*   **Layer F6 (Fully Connected Layer):**
    *   84 neurons.
    *   Connects to all 120 neurons in C5.
    *   Number of parameters: (120 + 1) * 84 = 10164.

*   **Output Layer (Output):**
    *   10 neurons (for 10 classes: digits 0-9).
    *   Uses a Euclidean Radial Basis Function (RBF) layer for classification, where each output unit calculates the Euclidean distance between its input vector and its parameter vector. This was a unique aspect of the original LeNet-5, often replaced by a softmax layer in modern implementations for probabilistic output.
    *   Number of parameters: (84 + 1) * 10 = 850 (if using simple linear connections to RBFs).

### Activation Function:

The original LeNet-5 used the `tanh` (hyperbolic tangent) activation function for its hidden layers, which was common at the time. Modern CNNs often use ReLU (Rectified Linear Unit) due to its computational efficiency and ability to mitigate vanishing gradients.

### Significance:

LeNet-5 demonstrated the power of CNNs for image recognition, establishing key architectural patterns (convolution-pooling sequence, hierarchical feature learning) that are still fundamental to deep learning today. Its success paved the way for more complex architectures like AlexNet, VGG, ResNet, and others.

In [1]:
import tensorflow as tf
from tensorflow.keras import layers, models

def build_lenet5(input_shape=(32, 32, 1), num_classes=10):
    model = models.Sequential([
        # C1: Convolutional Layer
        layers.Conv2D(6, kernel_size=(5, 5), activation='tanh', input_shape=input_shape, padding='valid'),
        # S2: Average Pooling Layer
        layers.AveragePooling2D(pool_size=(2, 2), strides=(2, 2), padding='valid'),

        # C3: Convolutional Layer
        layers.Conv2D(16, kernel_size=(5, 5), activation='tanh', padding='valid'),
        # S4: Average Pooling Layer
        layers.AveragePooling2D(pool_size=(2, 2), strides=(2, 2), padding='valid'),

        # C5: Fully Connected Layer (implemented as Conv2D for 1x1 output)
        layers.Conv2D(120, kernel_size=(5, 5), activation='tanh', padding='valid'), # Output will be 1x1x120

        # Flatten the output for the fully connected layers
        layers.Flatten(),

        # F6: Fully Connected Layer
        layers.Dense(84, activation='tanh'),

        # Output Layer: For classification (using softmax for probabilistic output)
        layers.Dense(num_classes, activation='softmax')
    ])
    return model

# Build the model
lenet_model = build_lenet5()

# Display the model summary
print("\n--- LeNet-5 Model Summary ---\n")
lenet_model.summary()

# Optional: Visualize the model (requires graphviz and pydot)
# tf.keras.utils.plot_model(lenet_model, show_shapes=True, show_layer_names=True, to_file='lenet5_model.png')
# from IPython.display import Image
# Image('lenet5_model.png')



--- LeNet-5 Model Summary ---



/usr/local/lib/python3.12/dist-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d (Conv2D)                 │ (None, 28, 28, 6)      │           156 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ average_pooling2d               │ (None, 14, 14, 6)      │             0 │
│ (AveragePooling2D)              │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 10, 10, 16)     │         2,416 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ average_pooling2d_1             │ (None, 5, 5, 16)       │             0 │
│ (AveragePooling2D)              │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_2 (Conv2D)               │ (None, 1, 1, 120)      │        48,120 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 120)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 84)             │        10,164 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 10)             │           850 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 61,706 (241.04 KB)

 Trainable params: 61,706 (241.04 KB)

 Non-trainable params: 0 (0.00 B)

In [2]:
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import (
    Conv2D,
    AveragePooling2D,
    Flatten,
    Dense
)

In [4]:
model=Sequential()
model.add(Conv2D(
    filters=6,
    kernel_size=(5,5),
    activation='tanh',
    input_shape=(32,32,1),
    padding='valid'
))
model.add(AveragePooling2D(
    pool_size=(2,2),
    strides=(2,2),
    padding='valid'
))
model.add(Conv2D(
    filters=16,
    kernel_size=(5,5),
    activation='tanh',
    padding='valid'
))
model.add(AveragePooling2D(
    pool_size=(2,2),
    strides=(2,2),
    padding='valid'
))

model.add(Flatten())

model.add(Dense(
    units=120,
    activation='tanh'
))
model.add(Dense(
    units=84,
    activation='tanh'
))
model.add(Dense(
    units=10,
    activation='softmax'
))

/usr/local/lib/python3.12/dist-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [5]:
model.summary()

Model: "sequential_2"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d_3 (Conv2D)               │ (None, 28, 28, 6)      │           156 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ average_pooling2d_2             │ (None, 14, 14, 6)      │             0 │
│ (AveragePooling2D)              │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_4 (Conv2D)               │ (None, 10, 10, 16)     │         2,416 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ average_pooling2d_3             │ (None, 5, 5, 16)       │             0 │
│ (AveragePooling2D)              │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten_1 (Flatten)             │ (None, 400)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 120)            │        48,120 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 84)             │        10,164 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_4 (Dense)                 │ (None, 10)             │           850 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 61,706 (241.04 KB)

 Trainable params: 61,706 (241.04 KB)

 Non-trainable params: 0 (0.00 B)

## Model Summary - Output Shape Flow

Let's break down how the output shape changes through each layer of the LeNet-5 model:

*   **Input Layer:**
    *   Input size: 32 × 32 × 1 (32x32 pixels, 1 channel for grayscale).

*   **First Convolutional Layer (C1):**
    *   **Filter size:** 5 × 5
    *   **Output filters:** 6
    *   **Calculation:** The output dimensions are determined by `(Input_Dim - Filter_Dim + 1)`. So, for a 32x32 input with a 5x5 filter and stride 1, the new dimensions are `(32 - 5 + 1) = 28`.
    *   **Output Shape:** 28 × 28 × 6

*   **First Pooling Layer (S2 - Average Pooling):**
    *   **Pool size:** 2 × 2
    *   **Stride:** 2
    *   **Calculation:** Pooling reduces dimensions by `Input_Dim / Pool_Size`. So, for a 28x28 input with a 2x2 pool, the new dimensions are `28 / 2 = 14`.
    *   **Output Shape:** 14 × 14 × 6

*   **Second Convolutional Layer (C3):**
    *   **Filter size:** 5 × 5
    *   **Output filters:** 16
    *   **Calculation:** For a 14x14 input with a 5x5 filter and stride 1, the new dimensions are `(14 - 5 + 1) = 10`.
    *   **Output Shape:** 10 × 10 × 16

*   **Second Pooling Layer (S4 - Average Pooling):**
    *   **Pool size:** 2 × 2
    *   **Stride:** 2
    *   **Calculation:** For a 10x10 input with a 2x2 pool, the new dimensions are `10 / 2 = 5`.
    *   **Output Shape:** 5 × 5 × 16

*   **C5 (Convolutional layer acting as Fully Connected):**
    *   **Filter size:** 5 × 5
    *   **Output filters:** 120
    *   **Calculation:** For a 5x5 input with a 5x5 filter and stride 1, the new dimensions are `(5 - 5 + 1) = 1`.
    *   **Output Shape:** 1 × 1 × 120

*   **Flatten Layer:**
    *   **Calculation:** This layer converts the 3D feature map into a 1D vector.
    *   **Output Shape:** 1 × 1 × 120 = 120 neurons

*   **F6 (Dense Layer):**
    *   **Output Units:** 84
    *   **Output Shape:** 84 neurons

*   **Output Layer (Dense Layer):**
    *   **Output Units:** 10 (for 10 classes)
    *   **Output Shape:** 10 neurons

In [6]:
model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

## Loading and Preprocessing `mnist_train_small.csv`

Since the `mnist_train_small.csv` file is available, we can load it to train our LeNet-5 model. This CSV usually contains the label in the first column and the pixel values in the subsequent columns. We need to perform similar preprocessing steps as with the `tf.keras.datasets.mnist` data:

1.  Load the CSV into a pandas DataFrame.
2.  Separate the labels from the pixel data.
3.  Reshape the pixel data into 28x28 images.
4.  Pad the 28x28 images to 32x32.
5.  Reshape to include the channel dimension (1 for grayscale).
6.  Normalize pixel values to [0, 1].

In [9]:
from tensorflow.keras.datasets import mnist
from tensorflow.keras.utils import to_categorical
import numpy as np

# Load dataset
(X_train, y_train), (X_test, y_test) = mnist.load_data()

# Resize to 32x32
X_train = tf.image.resize(
    X_train[..., np.newaxis],
    (32,32)
)

X_test = tf.image.resize(
    X_test[..., np.newaxis],
    (32,32)
)

# Normalize
X_train = X_train / 255.0
X_test = X_test / 255.0

11490434/11490434 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step


In [10]:
history = model.fit(
    X_train,
    y_train,
    epochs=10,
    batch_size=64,
    validation_split=0.2
)

Epoch 1/10
750/750 ━━━━━━━━━━━━━━━━━━━━ 34s 42ms/step - accuracy: 0.9145 - loss: 0.2937 - val_accuracy: 0.9585 - val_loss: 0.1407
Epoch 2/10
750/750 ━━━━━━━━━━━━━━━━━━━━ 29s 39ms/step - accuracy: 0.9675 - loss: 0.1082 - val_accuracy: 0.9724 - val_loss: 0.0935
Epoch 3/10
750/750 ━━━━━━━━━━━━━━━━━━━━ 32s 42ms/step - accuracy: 0.9778 - loss: 0.0729 - val_accuracy: 0.9768 - val_loss: 0.0776
Epoch 4/10
750/750 ━━━━━━━━━━━━━━━━━━━━ 39s 40ms/step - accuracy: 0.9836 - loss: 0.0542 - val_accuracy: 0.9783 - val_loss: 0.0702
Epoch 5/10
750/750 ━━━━━━━━━━━━━━━━━━━━ 29s 39ms/step - accuracy: 0.9865 - loss: 0.0441 - val_accuracy: 0.9786 - val_loss: 0.0694
Epoch 6/10
750/750 ━━━━━━━━━━━━━━━━━━━━ 30s 40ms/step - accuracy: 0.9893 - loss: 0.0347 - val_accuracy: 0.9831 - val_loss: 0.0593
Epoch 7/10
750/750 ━━━━━━━━━━━━━━━━━━━━ 30s 40ms/step - accuracy: 0.9911 - loss: 0.0282 - val_accuracy: 0.9819 - val_loss: 0.0637
Epoch 8/10
750/750 ━━━━━━━━━━━━━━━━━━━━ 34s 45ms/step - accuracy: 0.9925 - loss: 0.0231 - 

In [11]:
test_loss, test_accuracy = model.evaluate(X_test, y_test)

print("Test Accuracy:", test_accuracy)

313/313 ━━━━━━━━━━━━━━━━━━━━ 3s 11ms/step - accuracy: 0.9812 - loss: 0.0679
Test Accuracy: 0.9811999797821045
